# URL → Nine Ads (Krea 2 Identity Edit)

Same stack as your [face_swap_no_mask](https://colab.research.google.com/github/malihashar/headswap_V2/blob/face-swap-no-mask/notebooks/face_swap_no_mask.ipynb) Colab:

- clones `headswap_V2` (`face-swap-no-mask`)
- `setup_colab.sh --krea2` + **Drive model cache** (no HF token cell)
- uses `Krea2IdentityEditPipeline.edit_single_image` (real Identity Edit, one product photo + instruction)

**No Magic Hour API. No Ideogram. No `from_pretrained` HF login.**

### Before Run all
1. Runtime → **GPU** (A100 preferred; T4 works, slower)
2. Accept Drive mount when prompted
3. First run downloads ~18GB into Drive if missing (same as face-swap Colab)


In [ ]:
# @title 1) Setup — GPU · Drive · headswap_V2 · Krea2 weights
from pathlib import Path
import shutil
import subprocess
import sys
import os

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
BRANCH = "face-swap-no-mask"


def run(cmd, **kw):
    p = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if p.returncode != 0:
        print(p.stdout[-4000:])
        print(p.stderr[-4000:], file=sys.stderr)
        raise SystemExit(f"FAILED: {' '.join(map(str, cmd))}")
    return p


gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("No GPU. Runtime → Change runtime type → GPU, then re-run.")
print("GPU:", gpu.stdout.strip().splitlines()[0])

from google.colab import drive
drive.mount("/content/drive")

if REPO.exists() and not (REPO / ".git").is_dir():
    shutil.rmtree(REPO)

if not REPO.exists():
    print("Cloning repository…")
    run(["git", "clone", REPO_URL, str(REPO)])

print(f"Checking out {BRANCH}…")
run(["git", "-C", str(REPO), "fetch", "origin", BRANCH])
run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"])

env_py = REPO / "scripts" / "colab_env.py"
if not env_py.exists():
    raise SystemExit(f"Incomplete checkout: {env_py} missing. Delete {REPO} and re-run.")

print("Repo:", subprocess.getoutput(f"git -C {REPO} log --oneline -1"))
os.chdir(REPO)
print("Running setup_colab.sh --krea2 (uses Drive cache if present)…")
run(["bash", "scripts/setup_colab.sh", "--krea2"], cwd=str(REPO))

check = subprocess.run(
    [sys.executable, "-c", "import numpy, numpy._core.strings; print(numpy.__version__)"],
    capture_output=True, text=True,
)
if check.returncode != 0:
    print(check.stderr)
    raise SystemExit("numpy broken after setup — Runtime → Restart session, re-run this cell.")
print("numpy OK:", check.stdout.strip())

# thin deps for URL extract
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "beautifulsoup4", "lxml"])
print("Setup complete.")


In [ ]:
# @title 2) Config — paste product URL
from pathlib import Path

PRODUCT_URL = "https://satoshi-demo.myshopify.com/products/classic-straight-jeans"
# Optional (may be Cloudflare-blocked from Colab):
# PRODUCT_URL = "https://www.allbirds.com/products/mens-tree-gliders-natural-black-blizzard"

PRIMARY_IMAGE_INDEX = 0
NUM_ADS = 9
SEED = 46
OUT_DIR = Path("/content/url_ad_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FALLBACK_URL = "https://satoshi-demo.myshopify.com/products/classic-straight-jeans"
print("URL:", PRODUCT_URL)


In [ ]:
# @title 3) Extract product (JSON-LD / OpenGraph)
import json, io, requests
from typing import Any
from bs4 import BeautifulSoup
from PIL import Image
from IPython.display import display

UA = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}


def _as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def _price_from_offers(offers):
    for off in _as_list(offers):
        if not isinstance(off, dict):
            continue
        price = off.get("price") or off.get("lowPrice")
        cur = off.get("priceCurrency") or ""
        if price is not None:
            return f"{cur} {price}".strip()
    return None


def extract_product(url: str) -> dict[str, Any]:
    r = requests.get(url, headers=UA, timeout=25)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    brief = {
        "source_url": url,
        "product_name": None,
        "description": None,
        "price": None,
        "benefits": [],
        "reviews": [],
        "images": [],
        "cta_candidates": ["Shop now", "Buy now"],
        "extraction_gaps": [],
    }
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        raw = tag.string or tag.get_text() or ""
        try:
            data = json.loads(raw)
        except Exception:
            continue
        for node in (data if isinstance(data, list) else [data]):
            if not isinstance(node, dict):
                continue
            types = [str(t).lower() for t in _as_list(node.get("@type"))]
            if "product" not in types:
                continue
            brief["product_name"] = brief["product_name"] or node.get("name")
            brief["description"] = brief["description"] or node.get("description")
            for im in _as_list(node.get("image")):
                u = im if isinstance(im, str) else (im.get("url") if isinstance(im, dict) else None)
                if isinstance(u, str) and u.startswith("http") and u not in brief["images"]:
                    brief["images"].append(u)
            price = _price_from_offers(node.get("offers"))
            if price:
                brief["price"] = brief["price"] or price
            for rev in _as_list(node.get("review"))[:3]:
                if isinstance(rev, dict) and rev.get("reviewBody"):
                    brief["reviews"].append({
                        "quote": rev["reviewBody"],
                        "attribution": (
                            (rev.get("author") or {}).get("name")
                            if isinstance(rev.get("author"), dict)
                            else rev.get("author")
                        ),
                    })
    og_title = soup.find("meta", property="og:title")
    og_desc = soup.find("meta", property="og:description")
    og_img = soup.find("meta", property="og:image")
    if og_title and not brief["product_name"]:
        brief["product_name"] = og_title.get("content")
    if og_desc and not brief["description"]:
        brief["description"] = og_desc.get("content")
    if og_img:
        u = og_img.get("content")
        if u and u.startswith("http") and u not in brief["images"]:
            brief["images"].insert(0, u)
    for li in soup.select("li")[:40]:
        t = " ".join(li.get_text(" ", strip=True).split())
        if 20 <= len(t) <= 120 and t not in brief["benefits"]:
            brief["benefits"].append(t)
        if len(brief["benefits"]) >= 6:
            break
    if not brief["product_name"]:
        brief["extraction_gaps"].append("product_name")
    if not brief["images"]:
        brief["extraction_gaps"].append("images")
    if not brief["price"]:
        brief["extraction_gaps"].append("price")
    return brief


def download_image(url: str) -> Image.Image:
    resp = requests.get(url, headers=UA, timeout=30)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")


brief = None
last_err = None
for candidate in [PRODUCT_URL, FALLBACK_URL]:
    try:
        brief = extract_product(candidate)
        if brief.get("product_name") and brief.get("images"):
            PRODUCT_URL = candidate
            break
        last_err = f"thin extract for {candidate}: {brief.get('extraction_gaps')}"
    except Exception as e:
        last_err = f"{candidate}: {e}"
        brief = None

assert brief and brief.get("images"), f"extract failed: {last_err}"
idx = max(0, min(PRIMARY_IMAGE_INDEX, len(brief["images"]) - 1))
product_img = download_image(brief["images"][idx])
product_img.save(OUT_DIR / "product_primary.jpg", quality=95)
(OUT_DIR / "product.json").write_text(json.dumps(brief, indent=2), encoding="utf-8")

print(json.dumps({k: brief[k] for k in (
    "product_name", "price", "description", "images", "benefits", "reviews",
    "extraction_gaps", "source_url")}, indent=2)[:2500])
display(product_img.resize((384, 384)))


In [ ]:
# @title 4) Copy + 9 ad instructions (edit if needed)
NAME = brief.get("product_name") or "Product"
PRICE = brief.get("price") or ""
DESC = (brief.get("description") or "")[:280]
BENEFITS = brief.get("benefits") or []
REVIEWS = brief.get("reviews") or []
CTA = (brief.get("cta_candidates") or ["Shop now"])[0]

benefit_line = BENEFITS[0] if BENEFITS else (DESC.split(".")[0] if DESC else NAME)
review_line = REVIEWS[0]["quote"] if REVIEWS else "Customers love the fit and feel."

# Keep product identity; turn the photo into a finished square Meta/IG ad.
def ad_instruction(angle: str, headline: str, extra: str = "") -> str:
    return (
        f"Turn this exact product photo into a finished square Instagram/Meta ad. "
        f"Keep the product identical — same shape, materials, colors, packaging, logo. "
        f"Angle: {angle}. Add a large readable headline exactly: '{headline}'. "
        f"Add a clear CTA button with text exactly: '{CTA}'. "
        f"{extra} Clean commercial layout, feed-ready, no watermarks, no extra products."
    )

ANGLES = [
    ("01_benefit", ad_instruction("product benefit", benefit_line[:48])),
    ("02_problem_solution", ad_instruction("problem/solution", "Tired of settling?", f"Subtext about choosing {NAME}.")),
    ("03_review", ad_instruction("customer review / testimonial", review_line[:80])),
    ("04_feature", ad_instruction("feature callout", NAME, "Add up to three short feature callouts from the product.")),
    ("05_comparison", ad_instruction("comparison", f"Why {NAME}", "Simple before/after or vs-old layout.")),
    ("06_offer", ad_instruction("offer / price", PRICE or "Shop the drop")),
    ("07_social_proof", ad_instruction("social proof", "Loved by customers")),
    ("08_lifestyle", ad_instruction("lifestyle scene", NAME, "Place the same product naturally in a lifestyle setting.")),
    ("09_minimal_hero", ad_instruction("minimal product hero", NAME, "Soft studio background, lots of negative space.")),
]

assert len(ANGLES) == NUM_ADS
for k, p in ANGLES:
    print(k, "→", p[:110], "…")


In [ ]:
# @title 5) Load Krea2 Identity Edit (Drive cache) + generate 9 ads
from pathlib import Path
import importlib.util
import os
import time

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))
colab_env.ensure_import_path(REPO)

from headswap.config import load_config
from headswap.pipelines.krea2 import Krea2IdentityEditPipeline
from IPython.display import display, Markdown

cfg = load_config(str(REPO / "configs" / "krea2_identity_edit.yaml"))
cfg = dict(cfg)
cfg["seed"] = SEED
# single-image edit knobs (edit_single_image defaults)
cfg["single_edit_steps"] = 4
cfg["single_edit_cfg"] = 1.0
cfg["single_edit_denoise"] = 1.0
cfg["verbose"] = False

CACHE_DIR = Path("/content/.cache/url_ad_krea2")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Loading Krea2IdentityEditPipeline (warm models stay in memory)…")
pipe = Krea2IdentityEditPipeline(cfg=cfg, cache_dir=CACHE_DIR)

ads = []
t0 = time.perf_counter()
for i, (key, instruction) in enumerate(ANGLES):
    angle_dir = OUT_DIR / key
    angle_dir.mkdir(parents=True, exist_ok=True)
    # bump seed per angle for diversity while keeping product lock
    pipe.cfg["seed"] = SEED + i
    t = time.perf_counter()
    res = pipe.edit_single_image(product_img, instruction, out_dir=angle_dir)
    img = res["image"]
    img.save(OUT_DIR / f"{key}.png")
    ads.append((key, img))
    display(Markdown(f"### {key} · {time.perf_counter()-t:.0f}s"))
    display(img.resize((320, 320)))

print(f"Done {len(ads)} ads in {time.perf_counter()-t0:.0f}s total")
print("Saved under", OUT_DIR)


In [ ]:
# @title 6) 3×3 grid
from PIL import Image
from IPython.display import display

SIZE = 1024
grid = Image.new("RGB", (SIZE * 3, SIZE * 3), (255, 255, 255))
for i, (key, img) in enumerate(ads):
    r, c = divmod(i, 3)
    grid.paste(img.convert("RGB").resize((SIZE, SIZE)), (c * SIZE, r * SIZE))
grid_path = OUT_DIR / "grid_3x3.jpg"
grid.save(grid_path, quality=90)
print(grid_path)
display(grid.resize((768, 768)))


## Notes
- Uses the **same Drive model store** as face-swap (`/content/drive/MyDrive/headswap_V2/models`) — no HF token cell.
- First-ever Krea2 setup still downloads weights once via `setup_colab.sh` (same as face-swap).
- `edit_single_image` = upstream single-image Identity Edit graph (product photo + instruction).
- If extract fails on a brand PDP, the notebook falls back to the Shopify demo URL.
